In [1]:
%pip install pandas quinex click quinex-utils tdqm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!python3 -m spacy download en_core_web_md

     ---------------------------------------- 0.0/42.8 MB ? eta -:--:--
     - -------------------------------------- 1.6/42.8 MB 27.9 MB/s eta 0:00:02
     -------------- ------------------------ 15.5/42.8 MB 57.4 MB/s eta 0:00:01
     --------------------------- ----------- 30.4/42.8 MB 62.3 MB/s eta 0:00:01
     --------------------------------------  42.7/42.8 MB 61.8 MB/s eta 0:00:01
     ---------------------------------------- 42.8/42.8 MB 56.7 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python3.exe -m pip install --upgrade pip


In [3]:
%pip install git+https://github.com/FZJ-IEK3-VSA/llm-chunker.git

  Cloning https://github.com/FZJ-IEK3-VSA/llm-chunker.git to C:\Users\3265458R\AppData\Local\Temp\pip-req-build-sj7s2jl4
  Resolved https://github.com/FZJ-IEK3-VSA/llm-chunker.git to commit 392e686387bbd781ec591dd73819ea3025cbebc0
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Note: you may need to restart the kernel to use updated packages.


  Running command git clone --filter=blob:none --quiet https://github.com/FZJ-IEK3-VSA/llm-chunker.git 'C:\Users\3265458R\AppData\Local\Temp\pip-req-build-sj7s2jl4'

[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from quinex import Quinex
import pandas as pd
from tqdm import tqdm

DESCRIPTION_CSV = "cleaned_labeled_descriptions.csv"

description_df = pd.read_csv(DESCRIPTION_CSV)

quinex = Quinex()

c:\Users\3265458R\OneDrive - University of Glasgow\Documents\african_species\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



2026-10-30 13:59:01	Initializing quinex...


c:\Users\3265458R\OneDrive - University of Glasgow\Documents\african_species\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.



             .-----------------------.
            |   ___________________   |
            |  |   ,,,       ,,,   |  |
            |  |  ( ● )     ( ● )  |  |
            |  |   ```       ```   |  |
            |  |     `--___--´     |  |
            |  |___________________|  |
            |                         |
            | .---. .---. .---. .---. |
            | | 7 | | 8 | | 9 | | ÷ | |
            | '---' '---' '---' '---' |
            | .---. .---. .---. .---. |
            | | 4 | | 5 | | 6 | | x | |
            | '---' '---' '---' '---' |
            | .---. .---. .---. .---. |
            | | 1 | | 2 | | 3 | | - | |
            | '---' '---' '---' '---' |
            | .---. .---. .---. .---. |
            | | = | | 0 | | . | | + | |
            | '---' '---' '---' '---' |
             '-----------------------'                
                Q U I N E X v0.0.0
        
2026-10-30 13:59:10	✔ Pipeline initialized in 9.274 s.
2026-10-30 13:59:10	Note that using CPUs inste

In [5]:
import pandas as pd

# in case a key is missing in a nested dictionary, return None instead of raising an error
def safe_get(d, *keys):
    for key in keys:
        if not isinstance(d, dict):
            return None
        d = d.get(key)
    return d

import pandas as pd


def parse_quinex_results(results):
    # if Quinex returns no results, return an empty DataFrame
    if not results:
        return pd.DataFrame()

    rows = []

    for record in results:

        claim = record.get("claim") or {}

        entity = (claim.get("entity") or {}).get("text")
        property_name = (claim.get("property") or {}).get("text")

        quantity = claim.get("quantity") or {}

        quantity_text = quantity.get("text")

        normalized = quantity.get("normalized") or {}

        quantity_type = normalized.get("type")

        individual_quantities = (
            (normalized.get("individual_quantities") or {})
            .get("normalized", [])
        )

        values = []
        unit = None

        for q in individual_quantities:

            numeric_value = (
                ((q.get("value") or {})
                 .get("normalized") or {})
                .get("numeric_value")
            )

            if numeric_value is not None:
                values.append(numeric_value)

            if unit is None:
                unit_info = (q.get("unit") or {})
                normalized_units = unit_info.get("normalized") or []

                if normalized_units:
                    unit = normalized_units[0].get("text")
                else:
                    # fall back to raw unit text
                    text_info = unit_info.get("text") or {}

                    unit = (
                        text_info.get("suffixed")
                        or text_info.get("ellipsed")
                    )

        rows.append({
            "entity": entity,
            "property": property_name,
            "value_min": min(values) if values else None,
            "value_max": max(values) if values else None,
            "unit": unit,
            "quantity_text": quantity_text,
            "quantity_type": quantity_type,
        })

    return pd.DataFrame(rows)

In [6]:
#Sanity check
text = description_df[description_df["detected_language"] == "ENGLISH"]["cleaned_description"].iloc[110]

qclaims = quinex(text)

print(text)
print(parse_quinex_results(qclaims))


2026-10-30 13:59:11	📄 Applying pipeline to given text...
2026-10-30 13:59:11	Pre-processing done in 0.115 s.
2026-10-30 13:59:12	Identified 2 quantities in 1.221 s.
2026-10-30 13:59:19	Context analyzed in 7.256 s.
2026-10-30 13:59:19	✔ Done! Found 2 quantitative statements in 8.5 s.
Aristea juncifolia rarely reaches 300 mm in height and has terete, mostly unbranched stems and relatively large, boat-shaped, rhipidial spathes 12-18 mm long, green with broad transparent margins. The rigid, somewhat twisted leaves are elliptic or nearly round in section and without true margins.
                                  entity property  value_min  value_max unit  \
0                     Aristea juncifolia   height        300        300   mm   
1  large, boat-shaped, rhipidial spathes   length         12         18   mm   

  quantity_text    quantity_type  
0        300 mm  single_quantity  
1      12-18 mm            range  


In [1]:
import os
import pickle
import pandas as pd
from tqdm import tqdm

CHECKPOINT_DATA = "quinex_results.pkl"
CHECKPOINT_STATE = "quinex_state.pkl"
SAVE_EVERY = 100

In [ ]:
if os.path.exists(CHECKPOINT_DATA) and os.path.exists(CHECKPOINT_STATE):

    result_df = pd.read_pickle(CHECKPOINT_DATA)

    with open(CHECKPOINT_STATE, "rb") as f:
        start_idx = pickle.load(f)

    print(f"Resuming from row {start_idx:,}")

else:
    result_df = pd.DataFrame()
    start_idx = 0

english_descriptions = (
    description_df[description_df["detected_language"] == "ENGLISH"]
    .reset_index(drop=True)
)

for idx in tqdm(
    range(start_idx, len(english_descriptions)),
    total=len(english_descriptions),
    initial=start_idx,
):

    row = english_descriptions.iloc[idx]

    try:
        qclaims = quinex(row.cleaned_description)

        parsed_df = parse_quinex_results(qclaims)

        if not parsed_df.empty:
            parsed_df["ipni_id"] = row.ipni_id
            parsed_df["provider"] = row.provider

            result_df = pd.concat(
                [result_df, parsed_df],
                ignore_index=True,
            )

    except Exception as e:
        print(f"Error at row {idx}: {e}")

    if (idx + 1) % SAVE_EVERY == 0:

        result_df.to_pickle(CHECKPOINT_DATA)

        with open(CHECKPOINT_STATE, "wb") as f:
            pickle.dump(idx + 1, f)

result_df.to_pickle(CHECKPOINT_DATA)

print(f"Final checkpoint data saved to {CHECKPOINT_DATA}")

with open(CHECKPOINT_STATE, "wb") as f:
    pickle.dump(len(english_descriptions), f)

print(f"Final checkpoint state saved to {CHECKPOINT_STATE}")

Resuming from row 2,100


 29%|██▉       | 2100/7287 [00:00<?, ?it/s]


2026-10-30 13:59:19	📄 Applying pipeline to given text...
2026-10-30 13:59:19	Pre-processing done in 0.038 s.
2026-10-30 13:59:19	Identified 9 quantities in 0.181 s.


In [ ]:
result_df = pd.read_pickle("quinex_results.pkl")

result_df.head()

result_df.to_csv("extracted_quantities.csv", index=False)

,entity,property,value_min,value_max,unit,quantity_text,quantity_type,ipni_id,provider
0,relatively large corolla,length,28.0,32.0,mm,28-32 mm,range,77177570-1,Provided by: [ B ]. e-Flora of South Africa
1,large corolla,length,20.0,20.0,-25 (-27) mm,20-25(-27) mm,single_quantity,77177570-1,Provided by: [ B ]. e-Flora of South Africa
2,bracts,length,5.0,5.0,fid,5-fid,single_quantity,77177569-1,Provided by: [ B ]. e-Flora of South Africa
3,staminodes,length,0.7,0.7,mm,0.7 mm,single_quantity,770599-1,Provided by: [ A ]. e-Flora of South Africa
4,leaves,number of gland dots,2.0,2.0,rows of gland dots,two rows of gland dots,single_quantity,770599-1,Provided by: [ A ]. e-Flora of South Africa
